In [0]:
CREATE OR REPLACE VIEW ecom_dev.reporting.dim_date AS
SELECT date_key, full_date, year, quarter, month, month_name, year_month,
       week_of_year, day_of_month, day_of_week, day_name, is_weekend
FROM ecom_dev.gold.dim_date;

CREATE OR REPLACE VIEW ecom_dev.reporting.dim_customer AS
SELECT customer_key, customer_id, city, state, signup_date, valid_from, valid_to, is_current
FROM ecom_dev.gold.dim_customer;

CREATE OR REPLACE VIEW ecom_dev.reporting.dim_product AS
SELECT product_key, product_id, product_name, category, brand, unit_price, cost_price,
       is_active, valid_from, valid_to, is_current
FROM ecom_dev.gold.dim_product;

CREATE OR REPLACE VIEW ecom_dev.reporting.dim_seller AS
SELECT seller_key, seller_id, seller_name, city, state, seller_since, seller_rating
FROM ecom_dev.gold.dim_seller;

CREATE OR REPLACE VIEW ecom_dev.reporting.fact_sales AS
SELECT f.order_id, f.order_item_id, f.date_key, f.order_ts,
       f.customer_key, f.product_key, f.seller_key, f.order_status, f.channel,
       f.quantity, f.unit_price, f.discount_amount, f.freight_value,
       f.line_amount                                  AS net_sales,
       f.line_amount - f.quantity * p.cost_price      AS gross_margin
FROM ecom_dev.gold.fact_order_items f
LEFT JOIN ecom_dev.gold.dim_product p ON p.product_key = f.product_key;

CREATE OR REPLACE VIEW ecom_dev.reporting.fact_orders AS
SELECT order_id, date_key, customer_key, order_status, channel, purchase_ts, delivered_ts,
       estimated_delivery_date, items_count, items_amount, freight_amount, paid_amount,
       delivery_days, is_delivered, is_late
FROM ecom_dev.gold.fact_orders;

CREATE OR REPLACE VIEW ecom_dev.reporting.fact_payments AS
SELECT order_id, payment_sequential, date_key, payment_type, payment_installments, payment_value
FROM ecom_dev.gold.fact_payments;

CREATE OR REPLACE VIEW ecom_dev.reporting.fact_reviews AS
SELECT review_id, order_id, date_key, review_score, has_comment, review_ts
FROM ecom_dev.gold.fact_reviews;

SELECT order_status, count(DISTINCT order_id) AS orders, sum(net_sales) AS net_sales
FROM ecom_dev.reporting.fact_sales
GROUP BY order_status ORDER BY orders DESC;



CREATE OR REPLACE VIEW ecom_dev.reporting.fact_sales AS
SELECT f.order_id, f.order_item_id, f.date_key, f.order_ts,
       f.customer_key, f.product_key, f.seller_key, f.order_status, f.channel,
       f.quantity, f.unit_price, f.discount_amount, f.freight_value,
       f.line_amount                              AS net_sales,
       f.line_amount - f.quantity * p.cost_price  AS gross_margin,
       coalesce(f.order_status, '') <> 'canceled' AS is_revenue_order,
       f.order_status = 'returned'                AS is_returned
FROM ecom_dev.gold.fact_order_items f
LEFT JOIN ecom_dev.gold.dim_product p ON p.product_key = f.product_key;


-- One row per calendar day, including days with no sales (zeros).
-- This is also the input series for the forecasting model.
CREATE OR REPLACE VIEW ecom_dev.reporting.sales_daily AS
SELECT d.date_key, d.full_date, d.year_month, d.day_name, d.is_weekend,
       coalesce(a.orders, 0)       AS orders,
       coalesce(a.units, 0)        AS units,
       coalesce(a.net_sales, 0)    AS net_sales,
       coalesce(a.gross_margin, 0) AS gross_margin,
       coalesce(a.freight, 0)      AS freight
FROM ecom_dev.reporting.dim_date d
LEFT JOIN (
  SELECT date_key, count(DISTINCT order_id) AS orders, sum(quantity) AS units,
         sum(net_sales) AS net_sales, sum(gross_margin) AS gross_margin, sum(freight_value) AS freight
  FROM ecom_dev.reporting.fact_sales WHERE is_revenue_order GROUP BY date_key
) a ON a.date_key = d.date_key
WHERE d.full_date BETWEEN (SELECT min(to_date(order_ts)) FROM ecom_dev.reporting.fact_sales)
                      AND (SELECT max(to_date(order_ts)) FROM ecom_dev.reporting.fact_sales);

-- Month x category x brand
CREATE OR REPLACE VIEW ecom_dev.reporting.product_performance_monthly AS
SELECT d.year_month, p.category, p.brand,
       sum(s.quantity) AS units, sum(s.net_sales) AS net_sales, sum(s.gross_margin) AS gross_margin,
       round(100 * sum(s.gross_margin) / nullif(sum(s.net_sales), 0), 1) AS margin_pct,
       sum(CASE WHEN s.is_returned THEN s.quantity ELSE 0 END) AS returned_units
FROM ecom_dev.reporting.fact_sales s
JOIN ecom_dev.reporting.dim_date d    ON d.date_key = s.date_key
JOIN ecom_dev.reporting.dim_product p ON p.product_key = s.product_key
WHERE s.is_revenue_order
GROUP BY d.year_month, p.category, p.brand;

-- Month x channel: delivery and cancellation quality
CREATE OR REPLACE VIEW ecom_dev.reporting.delivery_performance_monthly AS
SELECT d.year_month, o.channel,
       count(*) AS orders,
       sum(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       sum(CASE WHEN o.is_delivered THEN 1 ELSE 0 END)              AS delivered_orders,
       round(avg(o.delivery_days), 1)                               AS avg_delivery_days,
       round(100 * sum(CASE WHEN o.is_late THEN 1 ELSE 0 END)
             / nullif(sum(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0), 1) AS late_pct
FROM ecom_dev.reporting.fact_orders o
JOIN ecom_dev.reporting.dim_date d ON d.date_key = o.date_key
GROUP BY d.year_month, o.channel;


SELECT 'days in sales_daily' AS check_name, CAST(count(*) AS DECIMAL(18,2)) AS value FROM ecom_dev.reporting.sales_daily
UNION ALL SELECT 'expected days (max - min + 1)',
       (SELECT datediff(max(to_date(order_ts)), min(to_date(order_ts))) + 1 FROM ecom_dev.reporting.fact_sales)
UNION ALL SELECT 'net_sales: daily view', (SELECT sum(net_sales) FROM ecom_dev.reporting.sales_daily)
UNION ALL SELECT 'net_sales: fact (revenue orders)', (SELECT sum(net_sales) FROM ecom_dev.reporting.fact_sales WHERE is_revenue_order)
UNION ALL SELECT 'net_sales: product view', (SELECT sum(net_sales) FROM ecom_dev.reporting.product_performance_monthly);

SELECT * FROM ecom_dev.reporting.sales_daily ORDER BY full_date DESC LIMIT 5;